# Toxic Comment Detection (Jupyter Notebook Edition)

This notebook reproduces the full toxic comment detection workflow—data loading, preprocessing, feature extraction, model training, hyper-parameter tuning, evaluation, and inference—in a single, self-contained environment. Run the notebook top-to-bottom inside Jupyter or JupyterLab to generate all artefacts and visualisations.


## 1. Environment Setup

The following cell imports every library used throughout the workflow, defines project-wide constants, and ensures that output folders exist for metrics and figures.


In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown
import ipywidgets as widgets
import re
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

RANDOM_STATE = 42
DATA_PATH = Path("train.csv")
ARTIFACTS_DIR = Path("artifacts")
RESULTS_DIR = Path("results")
EXTERNAL_TEST_PATH = Path("test.csv")
EXTERNAL_RESULTS_DIR = RESULTS_DIR / "external_test"

ARTIFACTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
EXTERNAL_RESULTS_DIR.mkdir(exist_ok=True)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

np.random.seed(RANDOM_STATE)


## 2. Load the Dataset

The Kaggle-style dataset ships with the repository as `train.csv`. It contains two columns: `comment_text` and the binary target `toxic`. We normalise the column names for convenience and perform a quick integrity check.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [c.strip().lower() for c in df_raw.columns]
if not {"comment_text", "toxic"}.issubset(df_raw.columns):
    raise ValueError("Expected 'comment_text' and 'toxic' columns in train.csv")

df_raw = df_raw[["comment_text", "toxic"]].copy()
df_raw["toxic"] = df_raw["toxic"].astype(int)
print(df_raw.shape)
df_raw.head()


## 3. Text Preprocessing Utilities

We replicate the five-stage cleaning pipeline from the original project: lower-casing, punctuation removal, tokenisation, stop-word removal, and lemmatisation. Each helper function is pure so that the transformations can be chained or reused individually.


In [ ]:
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()


def lowercase(text: str) -> str:
    return str(text).lower()


def remove_punctuation(text: str) -> str:
    return re.sub(r"[^a-z\s]", "", str(text))


def tokenize(text: str) -> List[str]:
    return str(text).split()


def remove_stopwords(tokens: Iterable[str]) -> List[str]:
    return [w for w in tokens if w and w not in STOP_WORDS]


def lemmatise(tokens: Iterable[str]) -> List[str]:
    return [LEMMATIZER.lemmatize(w) for w in tokens]


def join_tokens(tokens: Iterable[str]) -> str:
    return " ".join(tokens)


def clean_comment(text: str) -> Tuple[str, List[str]]:
    lowered = lowercase(text)
    no_punct = remove_punctuation(lowered)
    tokens = tokenize(no_punct)
    tokens = remove_stopwords(tokens)
    tokens = lemmatise(tokens)
    cleaned = join_tokens(tokens)
    return cleaned, tokens


def preprocess_comments(
    comments: Iterable[str],
    labels: Optional[Iterable[int]] = None,
) -> pd.DataFrame:
    processed_records = []
    lemmatised_tokens = []
    for comment in map(str, comments):
        cleaned, tokens = clean_comment(comment)
        processed_records.append(cleaned)
        lemmatised_tokens.append(tokens)
    data = {
        "comment": processed_records,
        "tokens": lemmatised_tokens,
    }
    if labels is not None:
        data["toxic"] = np.asarray(labels, dtype=int)
    return pd.DataFrame(data)


## 4. Run the Preprocessing Pipeline

This step materialises intermediate artefacts that mirror the original script. Besides the final cleaned text we also persist TF–IDF matrices and helper CSV files so that downstream analysis can reuse them if needed.


In [ ]:
preprocessed_df = preprocess_comments(
    df_raw["comment_text"],
    labels=df_raw["toxic"].values,
)

preprocessed_df.to_csv(ARTIFACTS_DIR / "final_preprocessed.csv", index=False)
preprocessed_df.head()


### Optional: External Test Set

If a `test.csv` file is present (with at least `comment_text` and optional `id`/`toxic` columns), the next cell will preprocess it so that baseline and tuned models can be compared on the held-out data.


In [ ]:
external_test_df = None
external_test_labels = None

if EXTERNAL_TEST_PATH.exists():
    external_raw = pd.read_csv(EXTERNAL_TEST_PATH)
    external_raw.columns = [c.strip().lower() for c in external_raw.columns]
    if "comment_text" not in external_raw.columns:
        raise ValueError("Expected 'comment_text' column in test.csv")
    if "id" not in external_raw.columns:
        external_raw["id"] = np.arange(len(external_raw))
    label_values = external_raw["toxic"].values if "toxic" in external_raw.columns else None
    external_test_df = preprocess_comments(
        external_raw["comment_text"],
        labels=label_values,
    )
    external_test_df.insert(0, "comment_text", external_raw["comment_text"].astype(str).values)
    external_test_df.insert(0, "id", external_raw["id"].values)
    if "toxic" in external_test_df.columns:
        external_test_labels = external_test_df["toxic"].values.astype(int)
    external_test_df.to_csv(ARTIFACTS_DIR / "external_test_preprocessed.csv", index=False)
    print(f"External test set loaded with {len(external_test_df)} rows.")
else:
    print(f"No external test set found at {EXTERNAL_TEST_PATH.resolve()}. Skipping external evaluation.")


We also inspect token counts to ensure the cleaning behaves as expected.


In [ ]:
word_counts = preprocessed_df["comment"].apply(lambda text: len(text.split()))
plt.figure(figsize=(8, 4))
plt.hist(word_counts, bins=50, color="#4C72B0")
plt.title("Word Count Distribution After Cleaning")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "wordcount_hist.png", bbox_inches="tight")
plt.show()


## 5. TF–IDF Feature Extraction

The notebook limits the vocabulary to 5,000 terms (minimum document frequency of 2) to remain faithful to the Python package version.


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, min_df=2)
X_tfidf = vectorizer.fit_transform(preprocessed_df["comment"].values)
y = preprocessed_df["toxic"].values

external_test_matrix = None
if external_test_df is not None:
    external_test_matrix = vectorizer.transform(external_test_df["comment"].values)
    sparse.save_npz(ARTIFACTS_DIR / "external_test_X_tfidf.npz", external_test_matrix)
    print("External test TF-IDF matrix shape:", external_test_matrix.shape)

sparse.save_npz(ARTIFACTS_DIR / "X_tfidf.npz", X_tfidf)
pd.DataFrame({"toxic": y}).to_csv(ARTIFACTS_DIR / "y.csv", index=False)

print("TF-IDF matrix shape:", X_tfidf.shape)


## 6. Train/Test Split

We stratify the split to preserve the 50/50 class balance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape


## 7. Evaluation Helpers

Utility functions for metric computation, ROC handling, confusion-matrix plotting, and the reusable training loop are defined below.


In [ ]:
def get_scores_for_roc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def compute_metrics(y_true, y_pred, y_score=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": np.nan,
    }
    if y_score is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            metrics["roc_auc"] = np.nan
    return metrics


def summarise_metrics(model_name, phase, metrics):
    row = {"model": model_name, "phase": phase}
    row.update(metrics)
    return row


def display_confusion_matrix(y_true, y_pred, title, ax=None):
    cm = confusion_matrix(y_true, y_pred)
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
        created_fig = True
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, values_format='d', cmap='Blues', colorbar=False)
    ax.set_title(title)
    ax.grid(False)
    if created_fig:
        fig.tight_layout()
        plt.show()
    return ax


def display_roc_curve(y_true, y_score, title, ax=None):
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
        created_fig = True
    if y_score is None:
        ax.axis('off')
        ax.text(0.5, 0.5, 'ROC curve not available', ha='center', va='center', fontsize=11)
        ax.set_title(title)
    else:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, label='Model', color='#4C72B0')
        ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Chance')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(title)
        ax.legend()
    if created_fig:
        fig.tight_layout()
        plt.show()
    return ax

## 8. Train Baseline and Tuned Models

We evaluate six classical algorithms. For each model we capture baseline metrics (default configuration) and tuned metrics using 3-fold cross-validation over the same hyper-parameter grids as the original scripts.


In [ ]:
model_specs = [
    {
        "name": "Logistic Regression",
        "short": "logreg",
        "factory": lambda: LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "param_grid": {
            "C": [0.1, 1.0, 5.0],
            "solver": ["liblinear", "lbfgs"],
            "penalty": ["l2"],
        },
    },
    {
        "name": "Multinomial Naive Bayes",
        "short": "nb",
        "factory": lambda: MultinomialNB(),
        "param_grid": {
            "alpha": [0.1, 1.0, 10.0],
            "fit_prior": [True, False],
        },
    },
    {
        "name": "SVM (LinearSVC)",
        "short": "svm",
        "factory": lambda: LinearSVC(random_state=RANDOM_STATE, dual=False),
        "param_grid": {
            "C": [0.5, 1.0, 2.0],
            "loss": ["squared_hinge"],
            "class_weight": [None, "balanced"],
        },
    },
    {
        "name": "Decision Tree",
        "short": "dt",
        "factory": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {
            "criterion": ["gini", "entropy"],
            "max_depth": [None, 20, 40],
            "min_samples_split": [2, 10, 25],
            "min_samples_leaf": [1, 5, 10],
        },
    },
    {
        "name": "Random Forest",
        "short": "rf",
        "factory": lambda: RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, n_jobs=-1),
        "param_grid": {
            "n_estimators": [200, 400],
            "max_depth": [None, 20, 40],
            "min_samples_split": [2, 10],
            "min_samples_leaf": [1, 5],
        },
    },
    {
        "name": "k-Nearest Neighbours",
        "short": "knn",
        "factory": lambda: KNeighborsClassifier(),
        "param_grid": {
            "n_neighbors": [3, 5, 11],
            "weights": ["uniform", "distance"],
            "metric": ["minkowski", "manhattan"],
        },
    },
]

all_metrics = []
all_best_estimators = {}
external_predictions = {} if external_test_matrix is not None else None

for spec in model_specs:
    print(f"=== {spec['name']} ===")

    results_by_phase = {}

    baseline_model = spec["factory"]()
    baseline_model.fit(X_train, y_train)
    y_pred = baseline_model.predict(X_test)
    y_score = get_scores_for_roc(baseline_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    all_metrics.append(summarise_metrics(spec["name"], "baseline", metrics))
    results_by_phase['baseline'] = {
        'metrics': metrics,
        'y_pred': y_pred,
        'y_score': y_score,
    }
    if external_predictions is not None:
        baseline_external_pred = baseline_model.predict(external_test_matrix)
        baseline_external_score = get_scores_for_roc(baseline_model, external_test_matrix)
        external_predictions.setdefault(spec["name"], {})['baseline'] = {
            'pred': baseline_external_pred,
            'score': baseline_external_score,
        }

    param_grid = spec["param_grid"]
    gs = GridSearchCV(
        spec["factory"](),
        param_grid,
        cv=3,
        n_jobs=-1,
        verbose=0,
    )
    gs.fit(X_train, y_train)
    tuned_model = gs.best_estimator_
    all_best_estimators[spec["name"]] = tuned_model
    print("Best hyper-parameters:", gs.best_params_)

    y_pred = tuned_model.predict(X_test)
    y_score = get_scores_for_roc(tuned_model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    all_metrics.append(summarise_metrics(spec["name"], "tuned", metrics))
    results_by_phase['tuned'] = {
        'metrics': metrics,
        'y_pred': y_pred,
        'y_score': y_score,
    }
    if external_predictions is not None:
        tuned_external_pred = tuned_model.predict(external_test_matrix)
        tuned_external_score = get_scores_for_roc(tuned_model, external_test_matrix)
        external_predictions.setdefault(spec["name"], {})['tuned'] = {
            'pred': tuned_external_pred,
            'score': tuned_external_score,
        }

    comparison_df = pd.DataFrame({
        phase: data['metrics'] for phase, data in results_by_phase.items()
    })
    comparison_df = comparison_df.loc[["accuracy", "precision", "recall", "f1", "roc_auc"]]
    comparison_df.columns = [phase.title() for phase in comparison_df.columns]
    display(comparison_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    display_confusion_matrix(
        y_test,
        results_by_phase['baseline']['y_pred'],
        f"{spec['name']} – Baseline",
        ax=axes[0],
    )
    display_confusion_matrix(
        y_test,
        results_by_phase['tuned']['y_pred'],
        f"{spec['name']} – Tuned",
        ax=axes[1],
    )
    fig.suptitle(f"{spec['name']} – Confusion Matrix Comparison")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    display_roc_curve(
        y_test,
        results_by_phase['baseline']['y_score'],
        f"{spec['name']} – Baseline ROC",
        ax=axes[0],
    )
    display_roc_curve(
        y_test,
        results_by_phase['tuned']['y_score'],
        f"{spec['name']} – Tuned ROC",
        ax=axes[1],
    )
    fig.suptitle(f"{spec['name']} – ROC Curve Comparison")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

metrics_df = pd.DataFrame(all_metrics)
metrics_df


The consolidated metrics table is saved to `results/` so that it can be inspected outside the notebook as well.


In [ ]:
metrics_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)
metrics_df


## 9. Model Comparison Visualisations

Bar plots help highlight which algorithms benefit most from tuning.


In [ ]:
comparison_table = metrics_df.pivot_table(
    index="model",
    columns="phase",
    values=["accuracy", "precision", "recall", "f1", "roc_auc"],
)
comparison_table = comparison_table.reindex(columns=['baseline', 'tuned'], level=1)
comparison_table.columns = [
    f"{metric.replace('_', ' ').title()} ({phase.title()})"
    for metric, phase in comparison_table.columns
]
comparison_table

metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 4))
    metric_data = metrics_df.pivot(index="model", columns="phase", values=metric)[['baseline', 'tuned']]
    metric_data.plot(kind="bar", ax=plt.gca())
    plt.title(f"{metric.replace('_', ' ').title()} – Baseline vs Tuned")
    plt.ylabel(metric.replace('_', ' ').title())
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{metric}_comparison.png", bbox_inches="tight")
    plt.show()

### External Test Predictions (Baseline vs Tuned)

When `test.csv` is available the table below summarises how each model's predictions change after hyper-parameter tuning. Individual CSV files containing per-comment predictions are saved to `results/external_test/`.


In [ ]:
if external_test_matrix is not None and external_predictions:
    summary_rows = []
    for spec in model_specs:
        name = spec["name"]
        pred_bundle = external_predictions.get(name)
        if not pred_bundle:
            continue
        baseline_pred = pred_bundle['baseline']['pred']
        tuned_pred = pred_bundle['tuned']['pred']
        summary = {
            "model": name,
            "baseline_positive_rate": float(np.mean(baseline_pred)),
            "tuned_positive_rate": float(np.mean(tuned_pred)),
            "prediction_shift": float(np.mean(tuned_pred) - np.mean(baseline_pred)),
            "prediction_agreement": float(np.mean(baseline_pred == tuned_pred)),
        }
        if external_test_labels is not None:
            baseline_metrics = compute_metrics(external_test_labels, baseline_pred, pred_bundle['baseline']['score'])
            tuned_metrics = compute_metrics(external_test_labels, tuned_pred, pred_bundle['tuned']['score'])
            for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
                summary[f"{metric}_baseline"] = baseline_metrics[metric]
                summary[f"{metric}_tuned"] = tuned_metrics[metric]
        summary_rows.append(summary)

        output_columns = {
            "id": external_test_df["id"],
            "comment_text": external_test_df["comment_text"],
            "clean_comment": external_test_df["comment"],
            "baseline_prediction": baseline_pred,
            "tuned_prediction": tuned_pred,
        }
        if pred_bundle['baseline']['score'] is not None:
            output_columns["baseline_score"] = pred_bundle['baseline']['score']
        if pred_bundle['tuned']['score'] is not None:
            output_columns["tuned_score"] = pred_bundle['tuned']['score']
        predictions_df = pd.DataFrame(output_columns)
        predictions_path = EXTERNAL_RESULTS_DIR / f"{spec['short']}_predictions.csv"
        predictions_df.to_csv(predictions_path, index=False)

    external_summary_df = pd.DataFrame(summary_rows)
    summary_path = EXTERNAL_RESULTS_DIR / "summary.csv"
    external_summary_df.to_csv(summary_path, index=False)
    display(external_summary_df)
    print(f"External predictions written to {EXTERNAL_RESULTS_DIR.resolve()}")
else:
    print("External test evaluation skipped (no test.csv detected).")


## 10. Interactive Inference Helper

Select the best tuned estimator (highest F1 score) and expose a convenience function for classifying arbitrary comments.


In [ ]:
# Utility helpers for single-comment inference across tuned models
def predict_single_comment(model_name: str, comment: str) -> Tuple[int, Optional[float]]:
    if not all_best_estimators:
        raise RuntimeError("No trained models detected. Run the training cells first.")
    if model_name not in all_best_estimators:
        raise ValueError(f"Unknown model: {model_name}")
    if not comment.strip():
        raise ValueError("Comment must not be empty.")
    cleaned, _ = clean_comment(comment)
    features = vectorizer.transform([cleaned])
    model = all_best_estimators[model_name]
    prediction = int(model.predict(features)[0])
    confidence: Optional[float] = None
    if hasattr(model, "predict_proba"):
        confidence = float(model.predict_proba(features)[0, 1])
    elif hasattr(model, "decision_function"):
        confidence = float(model.decision_function(features)[0])
    return prediction, confidence

model_names = [spec["name"] for spec in model_specs]
model_dropdown = widgets.Dropdown(
    options=model_names,
    description="Model:",
    disabled=False,
)
comment_box = widgets.Textarea(
    value="",
    placeholder="Type a comment to classify...",
    description="Comment:",
    layout=widgets.Layout(width='100%', height='120px'),
)
predict_button = widgets.Button(
    description="Predict toxicity",
    button_style='primary',
    icon='check',
)
output = widgets.Output()

def handle_predict(_):
    output.clear_output()
    try:
        comment = comment_box.value
        model_name = model_dropdown.value
        prediction, confidence = predict_single_comment(model_name, comment)
    except Exception as exc:
        with output:
            display(Markdown(f"**Error:** {exc}"))
        return
    label = "Toxic" if prediction else "Not toxic"
    with output:
        display(Markdown(f"**Prediction ({model_name}):** {label}"))
        if confidence is not None:
            display(Markdown(f"Confidence score: `{confidence:.3f}`"))

predict_button.on_click(handle_predict)

display(widgets.VBox([
    model_dropdown,
    comment_box,
    predict_button,
    output,
]))


The notebook is intentionally linear: rerun from the top whenever you modify preprocessing or parameter grids to keep every artefact in sync.
